In [17]:
#Ambiente

In [18]:
# Visualizando um estado:
import time
import gym_super_mario_bros
from nes_py.wrappers import JoypadSpace
from gym_super_mario_bros.actions import RIGHT_ONLY
import time
from OpenAIWrappers import nosso_wrapper
from stable_baselines3.common.vec_env import SubprocVecEnv
from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import CheckpointCallback

from stable_baselines3.common.callbacks import (
    CheckpointCallback,
    CallbackList
)

env = gym_super_mario_bros.make('SuperMarioBros-1-1-v0')
env = JoypadSpace(env, RIGHT_ONLY)
obs = env.reset()

for i in range(500):
    env.render()
    time.sleep(0.02)
    acao = 1
    obs, recompensa, terminou, info = env.step(acao)
    if terminou:
        print("Fase finalizada depois de {} steps".format(i+1))
        break
env.close()

Fase finalizada depois de 160 steps


In [19]:
#Treino

In [20]:
from stable_baselines3.common.callbacks import BaseCallback
from tqdm.auto import tqdm

class BarraProgressoCallback(BaseCallback):

    def _on_training_start(self):
        self.barra = tqdm(
            total=self.model._total_timesteps,
            desc="Treinamento da IA",
            unit=" steps"
        )

    def _on_step(self):
        self.barra.update(self.training_env.num_envs)
        self.barra.set_postfix({
            "realizados": f"{self.num_timesteps:,}"
        })
        return True

    def _on_training_end(self):
        self.barra.close()

In [21]:
import gym_super_mario_bros
from nes_py.wrappers import JoypadSpace
from gym_super_mario_bros.actions import RIGHT_ONLY
import time
from OpenAIWrappers import nosso_wrapper
from stable_baselines3.common.vec_env import SubprocVecEnv
from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import CheckpointCallback

from stable_baselines3.common.callbacks import (
    CheckpointCallback,
    CallbackList
)

def cria_env(nome_env):
    def _init():
        env = gym_super_mario_bros.make(nome_env)
        env = JoypadSpace(env, RIGHT_ONLY)
        env = nosso_wrapper(env)
        return env
    return _init


# Criar ambiente
threads_cpu = 16
env = SubprocVecEnv([cria_env(nome_env='SuperMarioBros-1-1-v0') for i in range(threads_cpu)])

# Callback para salvar checkpoints
checkpoint_callback = CheckpointCallback(
    save_freq=6250,
    save_path='modelos_salvos/',
    name_prefix='n1024b64l3'
)

# Callback de progresso
progresso_callback = BarraProgressoCallback()

# Combinar callbacks
callbacks = CallbackList([
    checkpoint_callback,
    progresso_callback
])

# Criar modelo
modelo = PPO(
    policy='CnnPolicy',
    env=env,
    gamma=0.99,
    n_steps=1024,
    batch_size=64,
    learning_rate=0.0003,
    vf_coef=0.5,
    ent_coef=0.01,
    verbose=1
)

inicio = time.time()

modelo.learn(
    total_timesteps=10000000,
    log_interval=10,
    callback=callbacks
)

fim = time.time()

print(
    'Tempo total do treinamento em horas:',
    (fim-inicio)/3600
)

env.close()


Using cpu device
Wrapping the env in a VecTransposeImage.


Treinamento da IA:   0%|          | 0/10000000 [00:00<?, ? steps/s]

KeyboardInterrupt: 

In [ ]:
#Visualização do Modelo

In [22]:
import gym_super_mario_bros
from nes_py.wrappers import JoypadSpace
from gym_super_mario_bros.actions import RIGHT_ONLY
import time
from OpenAIWrappers import nosso_wrapper
from stable_baselines3.common.vec_env import SubprocVecEnv
from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import CheckpointCallback

env = gym_super_mario_bros.make('SuperMarioBros-1-1-v0')
env = JoypadSpace(env, RIGHT_ONLY)
env = nosso_wrapper(env)

modelo = PPO.load('modelos_salvos/n1024b64l3_4600000_steps')

obs = env.reset()
recompensa_total = 0
terminou = False
while terminou!= True:
    env.render()
    time.sleep(0.04)
    acao, estado = modelo.predict(obs)
    obs, recompensa, terminou, info = env.step(acao.item())
    recompensa_total += recompensa
print('Recompensa total:', recompensa_total)
env.close()

Recompensa total: 300.0


In [23]:
import gym_super_mario_bros
from nes_py.wrappers import JoypadSpace
from gym_super_mario_bros.actions import RIGHT_ONLY
from OpenAIWrappers import nosso_wrapper

from stable_baselines3 import PPO
import cv2
import time
import os

# Ambiente
env = gym_super_mario_bros.make('SuperMarioBros-1-1-v0')
env = JoypadSpace(env, RIGHT_ONLY)
env = nosso_wrapper(env)

# Modelo treinado
modelo = PPO.load(
    'modelos_salvos/n1024b64l3_4600000_steps'
)

# Configurações
quantidade_runs = 5
pasta = 'videos_mario'

os.makedirs(pasta, exist_ok=True)

video = None

for run in range(quantidade_runs):

    print(f'Gravando tentativa {run + 1}/{quantidade_runs}')

    obs = env.reset()
    terminou = False
    recompensa_total = 0
    frames = 0

    while not terminou:

        # Captura a imagem do jogo
        frame = env.render(mode='rgb_array')

        if frame is not None:

            frame = cv2.cvtColor(
                frame,
                cv2.COLOR_RGB2BGR
            )

            if video is None:
                altura, largura, _ = frame.shape

                video = cv2.VideoWriter(
                    os.path.join(pasta, 'timelapse_mario.mp4'),
                    cv2.VideoWriter_fourcc(*'mp4v'),
                    30,
                    (largura, altura)
                )

            # Grava o frame
            video.write(frame)

        # IA escolhe a ação
        acao, estado = modelo.predict(
            obs,
            deterministic=True
        )

        obs, recompensa, terminou, info = env.step(
            acao.item()
        )

        recompensa_total += recompensa
        frames += 1

        # Limite de segurança para cada tentativa
        if frames >= 5000:
            break

    print('Recompensa:', recompensa_total)
    print('Frames:', frames)

    # Pequena pausa visual entre tentativas
    if video is not None:
        for _ in range(30):
            video.write(frame)

if video is not None:
    video.release()

env.close()

print('Vídeo finalizado!')

Gravando tentativa 1/5
Recompensa: 219.0
Frames: 2005
Gravando tentativa 2/5
Recompensa: 219.0
Frames: 2005
Gravando tentativa 3/5
Recompensa: 219.0
Frames: 2005
Gravando tentativa 4/5
Recompensa: 219.0
Frames: 2005
Gravando tentativa 5/5
Recompensa: 219.0
Frames: 2005
Vídeo finalizado!


In [24]:

import os
import json
import imageio
import numpy as np
import random
import numpy as np

from PIL import Image

# ==========================================
# 1. CAMINHOS
# ==========================================

PASTA = r"modelos_salvos\ghost_runs"

VIDEO_PRINCIPAL = os.path.join(PASTA, "captura_principal.mp4")
DADOS_PRINCIPAL = os.path.join(PASTA, "captura_principal.json")
DADOS_FANTASMAS = os.path.join(PASTA, "trajetorias.json")
SPRITE_PATH = os.path.join(PASTA, "mario_sprite.png")

VIDEO_FINAL = os.path.join(PASTA, "mario_com_fantasmas.mp4")

# ==========================================
# 2. CARREGAR DADOS
# ==========================================

with open(DADOS_PRINCIPAL, "r", encoding="utf-8") as f:
    principal = json.load(f)

with open(DADOS_FANTASMAS, "r", encoding="utf-8") as f:
    tentativas = json.load(f)

traj_principal = principal["trajetoria"]

# Quantidade de fantasmas desejada
QUANTIDADE_FANTASMAS = 100

# Opções: "ultimas", "aleatorias" ou "distribuidas"
MODO_SELECAO = "distribuidas"

# O Mario principal continua sendo a execução
# independente salva em captura_principal.json.

# Opcional: utilizar somente tentativas que falharam
candidatas = [
    t for t in tentativas
    if not t.get("concluiu", False)
]

if MODO_SELECAO == "ultimas":

    selecionadas = candidatas[-QUANTIDADE_FANTASMAS:]

elif MODO_SELECAO == "aleatorias":

    selecionadas = random.sample(
        candidatas,
        min(QUANTIDADE_FANTASMAS, len(candidatas))
    )

elif MODO_SELECAO == "distribuidas":

    indices = np.linspace(
        0,
        len(candidatas) - 1,
        min(QUANTIDADE_FANTASMAS, len(candidatas)),
        dtype=int
    )

    selecionadas = [candidatas[i] for i in indices]

else:
    raise ValueError("Modo de seleção inválido")

fantasmas = [
    tentativa["trajetoria"]
    for tentativa in selecionadas
]

print("Fantasmas selecionados:", len(fantasmas))
print("Mario principal: execução que concluiu a fase")


sprite_original = Image.open(SPRITE_PATH).convert("RGBA")

print("Passos da execução principal:", len(traj_principal))
print("Quantidade de fantasmas:", len(fantasmas))
print("Dimensões do sprite:", sprite_original.size)

# ==========================================
# 3. PREPARAR SPRITES TRANSLÚCIDOS
# ==========================================

OPACIDADE = 0.42

sprite_fantasma = sprite_original.copy()

alpha = np.array(sprite_fantasma.getchannel("A"))
alpha = (alpha * OPACIDADE).astype(np.uint8)

sprite_fantasma.putalpha(Image.fromarray(alpha))

# O recorte original começa 8 pixels à esquerda
# e 20 pixels acima da posição de referência.
OFFSET_X = 8
OFFSET_Y = 20

# ==========================================
# 4. FUNÇÃO PARA INSERIR UM FANTASMA
# ==========================================

def inserir_fantasma(frame, registro, camera_offset):

    x_mundo = registro["x_pos"]
    y_jogo = registro["y_pos"]

    # Converter coordenadas do mundo para a tela principal
    x_tela = int(x_mundo - camera_offset)

    # Conversão vertical calibrada
    y_tela = int(293 - y_jogo)

    # Origem do sprite
    x = x_tela - OFFSET_X
    y = y_tela - OFFSET_Y

    # Recortar caso o sprite ultrapasse as bordas
    sx0 = max(0, -x)
    sy0 = max(0, -y)
    sx1 = min(sprite_fantasma.width, frame.width - x)
    sy1 = min(sprite_fantasma.height, frame.height - y)

    if sx1 <= sx0 or sy1 <= sy0:
        return frame

    sprite_recortado = sprite_fantasma.crop(
        (sx0, sy0, sx1, sy1)
    )

    frame.alpha_composite(
        sprite_recortado,
        dest=(max(0, x), max(0, y))
    )

    return frame

# ==========================================
# 5. GERAR VÍDEO
# ==========================================

reader = imageio.get_reader(VIDEO_PRINCIPAL)

writer = imageio.get_writer(
    VIDEO_FINAL,
    fps=30,
    codec="libx264",
    ffmpeg_params=[
        "-crf", "15",
        "-preset", "slow",
        "-pix_fmt", "yuv420p"
    ],
    macro_block_size=16
)

try:
    total_frames = len(traj_principal)

    for i in range(total_frames):

        # Frame original do Mario principal
        frame_array = reader.get_data(i)

        frame = Image.fromarray(
            np.asarray(frame_array)
        ).convert("RGBA")

        registro_principal = traj_principal[i]

        # Deslocamento da câmera principal
        camera_offset = (
            registro_principal["x_pos"]
            - registro_principal["x_pos_screen"]
        )

        # Inserir cada fantasma
        for trajetoria in fantasmas:

            if i >= len(trajetoria):
                continue

            registro = trajetoria[i]

            frame = inserir_fantasma(
                frame,
                registro,
                camera_offset
            )

        # Converter frame para RGB
        frame_rgb = frame.convert("RGB")

        # Ampliar 4 vezes sem borrar os pixels
        frame_ampliado = frame_rgb.resize(
            (1024, 960),
            Image.Resampling.NEAREST
        )

        # Exportar em alta resolução
        writer.append_data(
            np.array(frame_ampliado)
        )

        if i % 100 == 0:
            print(f"Processando: {i}/{total_frames}")

finally:
    reader.close()
    writer.close()

print("\nVídeo final criado!")
print(VIDEO_FINAL)


Fantasmas selecionados: 4
Mario principal: execução que concluiu a fase
Passos da execução principal: 980
Quantidade de fantasmas: 4
Dimensões do sprite: (32, 28)
Processando: 0/980
Processando: 100/980
Processando: 200/980
Processando: 300/980
Processando: 400/980
Processando: 500/980
Processando: 600/980
Processando: 700/980
Processando: 800/980
Processando: 900/980

Vídeo final criado!
modelos_salvos\ghost_runs\mario_com_fantasmas.mp4
